# MESSAGEix-Pakistan 
### Current Measures Scenario
In this notebook, we are reading data and building a current measures scenerio.

<img src="https://wit.lums.edu.pk/sites/default/files/inline-images/WIT_Banner.jpg" alt="Girl in a jacket" width="850" height="250">

In [ ]:
import os
import sys
from pathlib import Path

# resolve the message_pak root (one level above this notebook's directory)
ROOT = Path(os.path.abspath('')).parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np

# fundamental libraries
import ixmp
import message_ix
from message_ix import log

from scripts.calibrate_t_d import model_calibrate
from scripts.reserve_margin import update_reserve_margin

# autoreload modules when changes are applied to them
%load_ext autoreload  
%autoreload all
%reload_ext autoreload
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Confirm root path (message_pak directory)
print('Project root:', ROOT)

Create the Scenario

In [ ]:
# creating ixmp platform object
new_mp = ixmp.Platform("committed 1.0")

# creating a new, empty scenario object
scenario = message_ix.Scenario(
    new_mp, 
    model="MESSAGEix-Pakistan", 
    scenario="current-measures", version="new"
)

Read Model Data

In [ ]:
data_path = ROOT / 'data' / 'model_file.xlsx'
scenario.read_excel(str(data_path), add_units=True, commit_steps=False, init_items=True)

In [ ]:
# scenario.check_out()
# model_calibrate(scenario, ["R12_PAK"], replace_negative=False)
update_reserve_margin(scenario, ["R12_PAK"], contin=0.15)

##### Solve the Model

In [ ]:
log.info(f"version number before commit(): {scenario.version}")
scenario.set_as_default()

# exporting the built model (Scenario) to GAMS with an optional case name
caseName = scenario.model + '__' + scenario.scenario + '__v' + str(scenario.version)

# solve model
scenario.solve(case=caseName)
obj_value = scenario.var("OBJ")["lvl"]
print(f"Objective value: {obj_value}")

##### Report the Results

In [ ]:
from report.legacy.iamc_report_hackathon import report
from datetime import datetime
import time

timestamp = datetime.now().strftime('%Y-%m-%d--%H-%M')
start = time.time()
out_dir = ROOT / 'output'
out_dir.mkdir(parents=True, exist_ok=True)
df, path_name = report(mp=new_mp, scen=scenario, out_dir=str(out_dir), out_file_timestamp=timestamp, IDEA_format=False)
end = time.time()
print(f'Reporting done in {end - start:.1f}s → {out_dir}')


In [ ]:
# close the connection to the database
new_mp.close_db()